In [1]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy




TO_A_ROUTE = f"{os.getcwd()}/../data/E01_toA/NEW"
EXCLUDED_TO_A = ["RESULTS", "E01_toA.zip"]

TIMESLOTS = [5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23]


TO_B_ROUTE = f"{os.getcwd()}/../data/E01_toB"
EXCLUDED_TO_B = ["RESULTS", "E01_toB.zip"]

In [2]:
final_df_toA = pd.DataFrame

final_df_toB = pd.DataFrame


dataframes_toA = []

# TOA
for csv_file in os.listdir(TO_A_ROUTE):
    if csv_file.endswith('.csv') and csv_file not in EXCLUDED_TO_A:
        file_path = os.path.join(TO_A_ROUTE, csv_file)
        
        auxiliar = pd.read_csv(file_path, encoding="utf-8", header=0)
        
        dataframes_toA.append(auxiliar)

if dataframes_toA:
    final_df_toA = pd.concat(dataframes_toA, ignore_index=True)
else:
    final_df_toA = pd.DataFrame()


STOPS_TOA = final_df_toA["Stop"].unique()


dataframes_toB = []

# TOA
for csv_file in os.listdir(TO_B_ROUTE):
    if csv_file.endswith('.csv') and csv_file not in EXCLUDED_TO_B:
        file_path = os.path.join(TO_B_ROUTE, csv_file)
        
        auxiliar = pd.read_csv(file_path, encoding="utf-8", header=0)
        
        dataframes_toB.append(auxiliar)

if dataframes_toB:
    final_df_toB = pd.concat(dataframes_toB, ignore_index=True)
else:
    final_df_toB = pd.DataFrame()


STOPS_TOB = final_df_toB["Stop"].unique()

In [3]:
def clean_dataframe(dataframe, daytype, direction):
    dataframe = dataframe[dataframe["DayType"] == daytype]
    dataframe = dataframe[dataframe["Direction"] == direction]
    dataframe = dataframe[dataframe["Occupation"] <= 80]
    dataframe = dataframe[dataframe["Occupation"] >= 0]
    dataframe = dataframe[dataframe["Board"] <= 80]
    dataframe = dataframe[dataframe["Board"] >= 0]
    dataframe = dataframe[dataframe["DropOff"] <= 80]
    dataframe = dataframe[dataframe["DropOff"] >= 0]
    dataframe = dataframe[dataframe["FreeDay"] == 0]

    return dataframe

In [4]:
final_df_toA = clean_dataframe(final_df_toA, "LA", 2)
final_df_toB = clean_dataframe(final_df_toB, "LA", 1)

In [5]:
#KPI SAVER PER STOP. ONLY GENERAL METRICS


kpi_toA = {}
kpi_toB = {}
#Ocupación media de los buses

kpi_toA["Mean_occupancy"] = (final_df_toA["Occupation"].mean()/80)*100

kpi_toB["Mean_occupancy"] = (final_df_toB["Occupation"].mean()/80)*100

#Media de buses fisicos encargados

each_day_unique_buses = []
each_day_total_trips_toA = []

for unique_day in final_df_toA["EventDate"].unique():
    that_day_data = final_df_toA[final_df_toA["EventDate"] == unique_day]
    each_day_total_trips_toA.append(len(that_day_data["TripId"].unique()))
    each_day_unique_buses.append(len(that_day_data["LogicVehicle"].unique()))

kpi_toA["Mean_unique_buses"] = np.mean(each_day_unique_buses)

each_day_unique_buses = []
print(f"Each daus toA{np.mean(each_day_total_trips_toA)}")
each_day_total_trips_toB = []

for unique_day in final_df_toB["EventDate"].unique():
    that_day_data = final_df_toB[final_df_toB["EventDate"] == unique_day]
    each_day_total_trips_toB.append(len(that_day_data["TripId"].unique()))
    each_day_unique_buses.append(len(that_day_data["LogicVehicle"].unique()))

kpi_toB["Mean_unique_buses"] = np.mean(each_day_unique_buses)
print(f"Each daus toB{np.mean(each_day_total_trips_toB)}")

kpi_toA["Mean_total_trips"] = np.mean(each_day_total_trips_toA) + np.mean(each_day_total_trips_toB)
kpi_toB["Mean_total_trips"] = np.mean(each_day_total_trips_toA) + np.mean(each_day_total_trips_toB)

print(f"GENERAL % OCC {(kpi_toA['Mean_occupancy'] + kpi_toB['Mean_occupancy'])/2}")

kpi_toA, kpi_toB

Each daus toA81.84810126582279
Each daus toB81.29746835443038
GENERAL % OCC 31.988866982558115


({'Mean_occupancy': np.float64(33.93928940759344),
  'Mean_unique_buses': np.float64(11.455696202531646),
  'Mean_total_trips': np.float64(163.14556962025318)},
 {'Mean_occupancy': np.float64(30.038444557522787),
  'Mean_unique_buses': np.float64(11.474683544303797),
  'Mean_total_trips': np.float64(163.14556962025318)})

In [6]:
import matplotlib.dates as mdates
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages # Herramienta clave para PDF multipágina

def project_time_of_departure(dataframe, timeslot, direction, pdf):
    # Añadimos .copy() para evitar el SettingWithCopyWarning de Pandas
    data_to_study = dataframe[dataframe["TimeSlot"] == timeslot].copy()

    # Convertimos a datetime
    data_to_study["EventTime"] = pd.to_datetime(data_to_study["EventTime"])

    # Asignamos la figura a una variable (fig) para poder guardarla
    fig = plt.figure(figsize=(10, 6))
    
    ax = sns.histplot(
        data=data_to_study, 
        x='EventTime', 
        bins=15, 
        kde=True,
        color='#1f77b4', 
        edgecolor='white'
    )
    
    formato_hora = mdates.DateFormatter('%H:%M')
    ax.xaxis.set_major_formatter(formato_hora)
    plt.xticks(rotation=45)

    # Etiquetas y título
    plt.title(f'Distribución de eventos - Franja {timeslot} (Sentido {direction})', fontsize=14, pad=15)
    plt.xlabel('Hora (HH:MM)', fontsize=12)
    plt.ylabel('Frecuencia', fontsize=12)

    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()

    # En lugar de mostrarla, la guardamos en el objeto PDF que le pasamos a la función
    pdf.savefig(fig)
    
    # Cerramos la figura para no saturar la memoria RAM
    plt.close(fig)
#toA

with PdfPages('histogramas_distribucion.pdf') as pdf:
    
    # Generamos las páginas para el sentido toA
    for timeslot in TIMESLOTS:
        project_time_of_departure(final_df_toA, timeslot, "toA", pdf)

    # Generamos las páginas para el sentido toB
    for timeslot in TIMESLOTS:
        project_time_of_departure(final_df_toB, timeslot, "toB", pdf)

/tmp/ipykernel_342277/1896279404.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data_to_study["EventTime"] = pd.to_datetime(data_to_study["EventTime"])
/tmp/ipykernel_342277/1896279404.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data_to_study["EventTime"] = pd.to_datetime(data_to_study["EventTime"])
/tmp/ipykernel_342277/1896279404.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data_to_study["EventTime"] = pd.to_datetime(data_to_study["EventTime"])
/tmp/ipykernel_342277/1896279404.py:10: UserWarning: Could not infer format, so each element will be parse

In [7]:
def salidas_por_franja(dataframe, direction):
    df = dataframe.copy()
    dias = df["EventDate"].unique()
    n_dias = len(dias)

    print(f"\n=== Salidas de cabecera por franja — sentido {direction} "
          f"({n_dias} dias) ===")
    print(f"{'franja':>6} {'viajes':>8} {'media/dia':>10} {'desv':>7}")

    filas = []
    for timeslot in TIMESLOTS:
        por_dia = (df[df["TimeSlot"] == timeslot]
                     .groupby("EventDate")["TripId"].nunique()
                     .reindex(dias, fill_value=0))
        media, desv = por_dia.mean(), por_dia.std(ddof=1)
        print(f"{timeslot:>6} {int(por_dia.sum()):>8} {media:>10.2f} {desv:>7.2f}")
        filas.append({"direccion": direction, "franja": timeslot,
                      "viajes": int(por_dia.sum()), "media_por_dia": media,
                      "desv": desv})

    res = pd.DataFrame(filas)
    print(f"{'TOTAL':>6} {res.viajes.sum():>8} {res.media_por_dia.sum():>10.2f}")
    return res


sal_toA = salidas_por_franja(final_df_toA, "toA")
sal_toB = salidas_por_franja(final_df_toB, "toB")
pd.concat([sal_toA, sal_toB]).to_csv("salidas_reales_por_franja.csv", index=False)


=== Salidas de cabecera por franja — sentido toA (158 dias) ===
franja   viajes  media/dia    desv
     5      364       2.30    0.82
     6      682       4.32    1.32
     7      924       5.85    1.74
     8      832       5.27    1.34
     9      675       4.27    1.22
    10      608       3.85    1.11
    11      666       4.22    1.20
    12      850       5.38    1.77
    13      848       5.37    1.71
    14      808       5.11    1.60
    15      848       5.37    1.55
    16      876       5.54    1.76
    17      806       5.10    1.49
    18      790       5.00    1.54
    19      728       4.61    1.78
    20      616       3.90    1.27
    21      484       3.06    1.15
    22      413       2.61    1.03
    23      114       0.72    0.46
 TOTAL    12932      81.85

=== Salidas de cabecera por franja — sentido toB (158 dias) ===
franja   viajes  media/dia    desv
     5      122       0.77    0.44
     6      526       3.33    0.91
     7      838       5.30    1.38
   

In [ ]:
# MAS ANALISIS

import math

from math import sqrt, exp, log

def central_moment(x, r):
    mean = np.nanmean(x)
    return sum((xi - mean) ** r for xi in x) / len(x)


def kurtosis(x):
    return (central_moment(x, 4) / central_moment(x, 2) ** 2) -3

def skewness(x):
    return central_moment(x, 3) / central_moment(x, 2)**(3/2)


def is_normal_simplified(variables:list):


    if len(variables) == 0:
        return np.bool_(False), [], 0, 0

    elif len(variables) < 50: #CASO Z score 1.96
        z_crit = 1.96
    elif len(variables) < 300: #CASO Z score 3.29
        z_crit = 3.29
    else:
        z_crit = "False"

    skew = skewness(variables)
    kurt = kurtosis(variables)


    n = len(variables)

    se_skew = math.sqrt(6.0 / n)
    se_kurt = math.sqrt(24.0 / n)

    mean = np.nanmean(variables)
    std_dev = math.sqrt(central_moment(variables, 2))
    z_skew = skew / se_skew
    z_kurt = kurt / se_kurt

    if z_crit != "False":
        approx_normal = (abs(z_skew) <= z_crit) and (abs(z_kurt) <= z_crit)

    else:
        approx_normal = (abs(skew) <= 2) and (abs(kurt) + 3 <= 7)

    return approx_normal, variables, mean, std_dev


def bhattacharyya_distance(mean_1, mean_2, std_1, std_2):
    return 0.25*((mean_1 -mean_2)**2/(std_1**2 + std_2**2)) + 0.5*(log((std_1**2 + std_2**2)/(2*(std_1 * std_2))))



def bhattacharyya_coefficient(mean_1, mean_2, std_1, std_2):
    return (sqrt(2*(std_1*std_2/(std_1**2 + std_2**2))))*exp(-(((mean_1 -mean_2)**2)/(4*(std_1**2 + std_2**2))))



normal_toA, vars_toA, mean_toA, std_toA = is_normal_simplified(final_df_toA['Occupation'])
normal_toB, vars_toB, mean_toB, std_toB = is_normal_simplified(final_df_toB['Occupation'])

combined_occupation = pd.concat([final_df_toA['Occupation'], final_df_toB['Occupation']])

occupation_toA = ((final_df_toA['Occupation'] / 80) * 100).to_numpy()

occupation_toB = ((final_df_toB['Occupation'] / 80) * 100).to_numpy()

# 2. Aplicamos la fórmula y lo convertimos a un array de NumPy
combined_occupation = ((combined_occupation / 80) * 100).to_numpy()


def histogram(data_hist, direction):
    plt.hist(data_hist, bins= 15)
    plt.xlabel(direction)
    plt.ylabel('Count')
    plt.title(f"Histograma de {direction}")
    plt.xticks(rotation=29)
    plt.grid(True, alpha=0.3)
    plt.show()


print(f" TOA Board {final_df_toA['Board'].sum()} df {final_df_toA['DropOff'].sum()}")
print(f" TOB Board {final_df_toB['Board'].sum()} df {final_df_toB['DropOff'].sum()}")

 TOA Board 798269 df 742841
 TOB Board 725807 df 713429
